# Solutions to the exercises of Chapter 1

Sample solutions for [Chapter 1, Why a variational principle](../../notebooks/part1_principle/01_why_a_variational_principle.ipynb).
Each solution gives the derivation and then checks the result by computation.

In [1]:
import numpy as np
import sympy as sp

try:
    import onsager as on
except ImportError:          # running from a clone without `pip install -e .`
    import sys
    sys.path.insert(0, "../..")
    import onsager as on

on.use_style()

## Exercise 1. A sphere sedimenting in a trap

With $U = Wx + \tfrac12kx^2$ the Rayleighian is

$$ R = \tfrac12\zeta\dot x^2 + (W + kx)\,\dot x , $$

and $\partial R/\partial\dot x = 0$ gives $\zeta\dot x = -(W + kx)$. Starting from
$x = 0$,

$$ x(t) = -\frac{W}{k}\big(1 - e^{-kt/\zeta}\big) , \qquad \dot x = -\frac{W}{\zeta}\,e^{-kt/\zeta} . $$

The rate of dissipation is $2\Phi = \zeta\dot x^2 = (W^2/\zeta)\,e^{-2kt/\zeta}$, and

$$ \int_0^\infty 2\Phi\,dt = \frac{W^2}{\zeta}\cdot\frac{\zeta}{2k} = \frac{W^2}{2k} . $$

The energy lost is $U(0) - U(-W/k) = 0 - \big(-W^2/k + W^2/2k\big) = W^2/2k$. The two
agree, as Eq. (1.3) requires at every instant.

In [2]:
t = sp.symbols("t", positive=True)
W, k, zeta = sp.symbols("W k zeta", positive=True)
x = sp.Function("x")
sol = sp.dsolve(sp.Eq(zeta * x(t).diff(t), -(W + k * x(t))), x(t), ics={x(0): 0}).rhs
heat = sp.integrate(zeta * sp.diff(sol, t) ** 2, (t, 0, sp.oo))
U = lambda X: W * X + k * X**2 / 2
print("x(t) =", sp.simplify(sol))
print("heat dissipated =", sp.simplify(heat), "   energy lost =", sp.simplify(U(0) - U(-W / k)))

x(t) = -W/k + W*exp(-k*t/zeta)/k
heat dissipated = W**2/(2*k)    energy lost = W**2/(2*k)


## Exercise 2. A circular tube

For $w = c\,(1 - r^2/a^2)$,

$$ \Phi = \frac{\eta}{2}\int_0^a \Big(\frac{dw}{dr}\Big)^2 2\pi r\,dr = \pi\eta\,c^2 , \qquad
   Q = \int_0^a w\,2\pi r\,dr = \frac{\pi a^2 c}{2} , $$

so $R = \pi\eta c^2 - \tfrac12\pi Ga^2c$, which is smallest at $c = Ga^2/4\eta$. Then
$Q = \pi Ga^4/8\eta$, the Hagen–Poiseuille law.

The result is exact because the exact solution of $\eta\nabla^2w = -G$ in a circle
is a paraboloid: it belongs to the trial family. A trial family that contains the
exact solution returns it, since nothing else in the family (or outside it) has a
lower Rayleighian.

In [3]:
r, a, c, eta, G = sp.symbols("r a c eta G", positive=True)
w = c * (1 - r**2 / a**2)
R = sp.integrate((eta / 2 * sp.diff(w, r) ** 2 - G * w) * 2 * sp.pi * r, (r, 0, a))
c_best = sp.solve(sp.diff(R, c), c)[0]
print("c =", c_best, "   Q =", sp.integrate(w.subs(c, c_best) * 2 * sp.pi * r, (r, 0, a)))
print("residual of the Stokes equation:", sp.simplify(eta * (sp.diff(r * sp.diff(w, r), r) / r).subs(c, c_best) + G))

c = G*a**2/(4*eta)    Q = pi*G*a**4/(8*eta)
residual of the Stokes equation: 0


## Exercise 3. A rectangular duct

In units of $a$ and $Ga^2/\eta$, with $\phi = (1 - x^2)(1 - y^2/b^2)$ on
$-1 < x < 1$, $-b < y < b$:

$$ \int\phi\,dS = \frac43\cdot\frac{4b}{3} = \frac{16b}{9} , \qquad
   \int(\nabla\phi)^2 dS = \frac83\cdot\frac{16b}{15} + \frac{16}{15}\cdot\frac{8}{3b} = \frac{128}{45}\Big(b + \frac1b\Big) . $$

The flow rate of the optimised trial field is the square of the first divided by
the second (Section 1.3),

$$ Q_{\rm trial} = \frac{(16b/9)^2}{\tfrac{128}{45}(b + 1/b)} = \frac{10}{9}\,\frac{b^3}{b^2 + 1} . $$

In [4]:
x, y, b = sp.symbols("x y b", positive=True)
phi = (1 - x**2) * (1 - y**2 / b**2)
num = sp.integrate(phi, (x, -1, 1), (y, -b, b))
den = sp.integrate(sp.diff(phi, x) ** 2 + sp.diff(phi, y) ** 2, (x, -1, 1), (y, -b, b))
Q_trial = sp.simplify(num**2 / den)
print("Q_trial =", sp.factor(Q_trial))
print("\n   b     trial      exact     error")
for bb in (1, 2, 4, 10):
    qt, qe = float(Q_trial.subs(b, bb)), on.duct_flow_rate(bb)
    print(f"  {bb:3d}   {qt:7.4f}   {qe:8.4f}   {100 * (qt / qe - 1):+5.1f} %")
print(f"limit of a very flat duct: trial/exact -> (10/9)/(4/3) = {10 / 9 / (4 / 3):.4f}")

Q_trial = 10*b**3/(9*(b**2 + 1))

   b     trial      exact     error
    1    0.5556     0.5623    -1.2 %
    2    1.7778     1.8295    -2.8 %
    4    4.1830     4.4930    -6.9 %
   10   11.0011    12.4930   -11.9 %
limit of a very flat duct: trial/exact -> (10/9)/(4/3) = 0.8333


The single trial function becomes worse as the duct flattens: $-1.2$ %, $-2.8$ %,
$-6.9$ %, and $-16.7$ % in the limit $b \to \infty$. In a flat duct the velocity
across the wide direction is nearly uniform, falling to zero only within a distance
of order $a$ of the side walls. The trial function forces a parabola in $y$, which
is far too rounded. Better choices are a flatter profile in $y$, such as
$1 - (y/b)^{2m}$ with $m$ large or $1 - \cosh(\kappa y)/\cosh(\kappa b)$ with
$\kappa$ a second parameter, or simply more polynomial terms, as in Section 1.3.

## Exercise 4. Not a minimum of dissipation

Take a field of fixed shape, $w = s\,\phi(x, y)$, with amplitude $s$. Write
$K = \int(\nabla\phi)^2dS$ and $\beta = \int\phi\,dS$. The condition that the field
dissipates energy at the rate at which it is supplied, $\eta s^2K = Gs\beta$, fixes
the amplitude, $s = G\beta/\eta K$, and then

$$ Q = s\beta = \frac{G}{\eta}\,\frac{\beta^2}{K} . $$

On the other hand, minimising $R = \tfrac12\eta s^2K - Gs\beta$ over $s$ gives the same
amplitude and $R_{\min} = -G^2\beta^2/2\eta K = -\tfrac12GQ$. Therefore the shape that
makes $R$ smallest is the shape that makes $Q$ largest: among fields that balance
their energy budget, the real flow carries the most. It dissipates more than any of
the others, not less. "Least dissipation" is true only in the sense of Helmholtz's
theorem, where the flow rate (or the boundary velocity) is held fixed.

The cell checks this for the square duct: random combinations of six trial
functions never exceed the flow rate of the optimal combination, which itself lies
below the exact value.

In [5]:
nodes, weights = np.polynomial.legendre.leggauss(30)
X, Y = np.meshgrid(nodes, nodes)
W2 = np.outer(weights, weights)
basis = []
for i in range(3):
    for j in range(3 - i):
        f = (1 - X**2) * (1 - Y**2) * X ** (2 * i) * Y ** (2 * j)
        fx = (-2 * X * X ** (2 * i) + (1 - X**2) * (2 * i * X ** (2 * i - 1) if i else 0)) * (1 - Y**2) * Y ** (2 * j)
        fy = (-2 * Y * Y ** (2 * j) + (1 - Y**2) * (2 * j * Y ** (2 * j - 1) if j else 0)) * (1 - X**2) * X ** (2 * i)
        basis.append((f, fx, fy))
K = np.array([[np.sum(W2 * (p[1] * q[1] + p[2] * q[2])) for q in basis] for p in basis])
beta = np.array([np.sum(W2 * p[0]) for p in basis])
a_opt = np.linalg.solve(K, beta)
Q_opt = beta @ a_opt
rng = np.random.default_rng(1)
Q_random = [(beta @ a) ** 2 / (a @ K @ a) for a in a_opt * (1 + 0.5 * rng.standard_normal((2000, 6)))]
print(f"optimal combination: Q = {Q_opt:.6f}   exact: {on.duct_flow_rate():.6f}")
print(f"2000 random combinations: largest Q = {max(Q_random):.6f}, smallest = {min(Q_random):.6f}")

optimal combination: Q = 0.562229   exact: 0.562308
2000 random combinations: largest Q = 0.562223, smallest = 0.000414
